# Milestone 3 · E1 — Công cụ gán nhãn dùng chung cho nhóm 4 người

**Không sửa code, không thêm biến ROOT/PROJECT_ROOT, không tạo notebook riêng cho từng người.**
Chạy Cell 1 → Cell 2, điền biểu mẫu và bấm **Kiểm tra & mở bảng**.
Cell 3 kiểm tra tiến độ; Cell 4 đóng bảng và chỉ ra file cần bàn giao.

Mỗi người dùng cùng notebook nhưng một phiên Colab riêng. Đường dẫn được chọn trong biểu mẫu,
không điền vào các cell Python. Dữ liệu có thể nằm trên Drive/máy riêng của mỗi người.
Notebook không cào lại, không parse lại, không tự chạy auto hay tự chia tập toàn nhóm.

**Tên và mã nhận diện do người dùng tự nhập; không có danh sách người dùng điền sẵn.**
Tên có thể dùng tiếng Việt có dấu. Khi bắt đầu mới, mã lưu tiến độ có thể để trống để công cụ
tạo mã không dấu từ tên. Nếu nhiều người có tên tạo ra cùng mã, tự chọn mã riêng để tránh trùng.
Khi tiếp tục nhãn cũ, nhập đúng mã đã dùng hoặc nạp cấu hình cũ, không đổi tên file tiến độ.

Một phần manual cần **5 bài × ít nhất 20 nhãn xác nhận/bài**. Bỏ qua chưa được tính là nhãn.
Nhóm 4 người cần đối soát ít nhất 20 bài manual khác nhau và 400 nhãn; việc gộp corpus,
kiểm tra quota auto và partition sẽ thực hiện sau, không tự kết luận đạt trong notebook này.

## Cell 1 — Chuẩn bị môi trường · chỉ bấm chạy

**Colab:** chọn runtime CPU, chạy cell và mount tài khoản Drive chứa dữ liệu khi được yêu cầu.
Code M3_E1 được tải/cập nhật vào runtime; dependency của bảng được cài tự động.
Mở notebook từ GitHub chưa tự tải source vào runtime, nên không bỏ qua cell này.

**Máy local:** mở notebook từ thư mục repo hoặc notebooks/; cell nhận diện code và cài thư viện
nếu còn thiếu. Không dùng ổ D của máy cá nhân làm đường dẫn cho Colab.

Nếu đang có bảng mở, đóng phiên ở Cell 4 trước khi chạy lại Cell 1. Sau reset runtime,
chạy lại từ Cell 1; nhãn đã lưu trên Drive vẫn còn. Cell dùng git pull --ff-only,
không force/reset hay xóa các thay đổi source đang có.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib
import importlib.util

if 'tool' in globals():
    tool.close()

try:
    import google.colab
    on_colab = True
except ImportError:
    on_colab = False

if on_colab:
    from google.colab import drive, output as colab_output
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    code_directory = Path('/content/arxiv_m3_e1')
    if not code_directory.exists():
        subprocess.run(['git', 'clone', '--branch', 'M3_E1', '--single-branch',
            'https://github.com/NewB-S2deeC0de/arXiv.git', str(code_directory)], check=True)
    else:
        subprocess.run(['git', '-C', str(code_directory), 'pull', '--ff-only',
                        'origin', 'M3_E1'], check=True)
else:
    code_directory = next((p for p in (Path.cwd(), *Path.cwd().parents)
                           if (p / 'src/matching/notebook_tool.py').is_file()), None)
    if code_directory is None:
        raise RuntimeError('Mở notebook từ repo M3_E1 hoặc thư mục notebooks của repo đã cập nhật.')

if not (code_directory / 'src/matching/notebook_tool.py').is_file():
    raise RuntimeError('Source còn cũ. Cập nhật nhánh M3_E1 có notebook_tool.py rồi chạy lại.')

missing_packages = any(importlib.util.find_spec(name) is None
                       for name in ('pyarrow', 'ipywidgets', 'IPython'))
if on_colab or missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
        str(code_directory / 'notebooks/requirements-labeling.txt')], check=True)

sys.path.insert(0, str(code_directory / 'src'))
importlib.invalidate_caches()
for name in ('matching.labeling', 'matching.auto_matcher', 'matching.dataset_plan',
             'matching.labeling_table', 'matching.notebook_tool'):
    module = importlib.import_module(name)
    importlib.reload(module)

from IPython.display import display
from matching.notebook_tool import AnnotationNotebook

if on_colab:
    colab_output.enable_custom_widget_manager()

print('Đã chuẩn bị công cụ. Chạy Cell 2 để mở biểu mẫu, không cần sửa cell này.')

## Cell 2 — Mở biểu mẫu, chọn dữ liệu và bắt đầu gán nhãn

Chạy cell, sau đó **điền trực tiếp trong biểu mẫu**. Không thay bất kỳ dòng Python nào.

### Chọn công việc

| Công việc | Khi nào chọn? | Cần chuẩn bị |
|---|---|---|
| **Bắt đầu manual trên dữ liệu riêng** | Chưa có nhãn trên dữ liệu này | BibTeX M2, references M1 và thư mục được phép lưu nhãn. Kế hoạch 5 bài × 20 entry tự tạo và lưu lại. |
| **Tiếp tục manual với kế hoạch gốc** | Đã có nhãn từ notebook/CLI trước đây | Hai bảng gốc, file kế hoạch JSON gốc và đúng thư mục cha chứa manual/ đã lưu. |
| **Review kết quả auto** | Cần xử lý các entry needs_review | Hai bảng của snapshot auto, thư mục kết quả auto hoàn tất và thư mục lưu nhãn. Không cần kế hoạch manual. |
| **Thử giao diện bằng dữ liệu giả** | Muốn thử các nút trước | Không cần điền tên/đường dẫn; demo không ghi vào tiến độ thật. |

### Điền thông tin bằng biểu mẫu

1. **Tên người gán nhãn:** tự nhập tên thật hoặc tên nhận diện của bạn, có thể có dấu/khoảng trắng.
   **Mã lưu tiến độ:** người mới để trống để tạo mã từ tên, hoặc tự nhập mã riêng chỉ gồm chữ
   không dấu, số, _ hoặc -. Mã dùng cho tên file và phân công, không phải tài khoản Drive.
   Người đã có nhãn phải giữ mã cũ trong kế hoạch (ví dụ member1); tên reviewer cũ trong các
   nhãn đã lưu không bị đổi. Nạp cấu hình cũ là cách an toàn nhất để tiếp tục.
2. **BibTeX M2:** chọn file bibtex.parquet đã parse. Có thể chọn thư mục shards của riêng bảng BibTeX.
3. **References M1:** chọn references.parquet trong snapshot export đã chốt, hoặc thư mục shards riêng bảng.
4. **Thư mục lưu nhãn:** chọn thư mục có quyền ghi trên Drive của bạn. Công cụ tự tạo manual/,
   plans/, notebook_profiles/ bên trong. Để tiếp tục nhãn cũ, chọn đúng thư mục **cha chứa manual/**;
   không chọn file nhãn .parquet và không chọn chính thư mục manual/.
5. **Kế hoạch gốc JSON:** chỉ hiện với tiếp tục manual cũ. Chọn kế hoạch từng dùng cho nhãn đó.
6. **Thư mục kết quả auto:** chỉ hiện với review; phải có _SUCCESS, summary.json và auto_labels.parquet.
7. **DOI bổ sung (tùy chọn)** nằm ngay dưới References M1 trên biểu mẫu: chọn doi_index.parquet
   để hiện DOI bổ sung; có thể để trống. Không cần mở mục thu gọn để tìm ô DOI.
   Bấm **Hiện tùy chọn nâng cao** khi cần: cá nhân mới có thể chỉ định 5 ID,
   loại bài đã thuộc auto/người khác hoặc chọn checkpoint M2 và
   reference_snapshots.parquet M1 để lọc bài thiếu/lỗi. Không có báo cáo sẽ hiện giới hạn kiểm tra.

### Dùng nút Chọn… hoặc dán Copy path

Nút **Chọn…** mở trình duyệt thư mục ngay trong notebook: chọn folder → Mở thư mục → chọn file →
Dùng lựa chọn. Với nơi lưu nhãn, có thể dùng thư mục hiện tại. Công cụ chỉ duyệt từng thư mục,
không quét đệ quy cả Drive. Nếu thư mục có quá nhiều mục, dán đường dẫn file trực tiếp.

Cách khác: Files bên trái Colab → drive → MyDrive → tìm file → Copy path → dán vào ô.
MyDrive là Drive của tôi của tài khoản đã mount. Không dán link https://drive.google.com/…;
link chia sẻ không tự trở thành đường dẫn filesystem. Thư mục mới để ghi có thể điền trực tiếp;
thư mục đó sẽ được tạo khi chuẩn bị kế hoạch/tiến độ.

Các thành viên có thể có đường dẫn khác nhau. Nếu dùng dữ liệu khác nội dung thì cần kế hoạch
cá nhân riêng; không dùng file kế hoạch của người khác cho snapshot khác và không bỏ kiểm tra fingerprint.
Nguồn DOI chỉ bổ sung metadata hiển thị, không đổi pool hoặc chọn sẵn dự đoán auto.

### Bấm Kiểm tra & mở bảng

Chờ kiểm tra file và dựng pool. Nếu báo lỗi, sửa **ô biểu mẫu được nêu tên** rồi bấm lại;
không thêm cell import, ROOT hay Path. Khi đổi người/đường dẫn/công việc, bảng cũ được đóng
để tránh ghi nhãn theo cấu hình cũ; các quyết định đã lưu vẫn được giữ.

**Lần sau:** cấu hình được lưu tại thư mục nhãn/notebook_profiles/<mã_lưu>.json (nếu giữ dấu chọn).
Chọn file đó ở ô File cấu hình đã lưu → Nạp cấu hình → kiểm tra người/đường dẫn → Kiểm tra & mở bảng.
Nạp cấu hình không tự ghi nhãn. Một người/file tiến độ tại một thời điểm, nhất là trên Drive.

In [ ]:
if 'AnnotationNotebook' not in globals():
    raise RuntimeError('Chạy Cell 1 trước để chuẩn bị công cụ.')
if 'tool' in globals():
    tool.close()

start_directory = Path('/content/drive/MyDrive') if on_colab else Path.cwd()
tool = AnnotationNotebook(group_members=4, start_directory=start_directory)
display(tool.root)

## Cách thao tác trên bảng

- Đọc entry ở phía trên; tìm từ đặc trưng trong title/tác giả/DOI/arXiv ID rồi Enter hoặc rời ô.
- Bấm **Chọn** trên dòng reference khớp: lưu ngay rồi chuyển entry chưa gán. Không cần nhập số.
- Bấm **Xem tất cả** để bỏ lọc; đọc đủ các trang trước khi xác nhận **Không có match**.
  Nút no-match bị khóa khi đang lọc. Tìm kiếm không có kết quả không phải bằng chứng no-match.
- **Chưa chắc / Tiếp** không tạo nhãn, không tính quota. Thiếu metadata thì đọc nguồn hoặc để lại;
  không đoán nhãn âm. **Entry chưa gán** quay lại các ca pending.
- Có thể quay lại sửa/xóa nhãn, chọn trực tiếp bài/entry; tác giả dài có thể mở rộng, link arXiv mở nguồn.
- DOI preprint arXiv và DOI bản tạp chí có thể khác; giao diện giữ mọi giá trị để bạn kiểm tra.
- Gán manual độc lập, không xem dự đoán auto trước. Bảng không xếp hạng ứng viên bằng matcher.

Người mới chọn 5 bài manual **trước khi chạy auto**, rồi loại các bài đó khỏi tập auto.
Kế hoạch cá nhân lưu một lần tại plans/<mã_lưu>.json; lần sau không chọn lại ngẫu nhiên.
Đã có nhãn nhưng thiếu kế hoạch cá nhân thì công cụ yêu cầu tiếp tục bằng kế hoạch gốc.
Nếu đã bỏ qua một số entry, cần xác minh lại hoặc bổ sung kế hoạch có đối soát để đủ 20 nhãn/bài;
không sửa tùy tiện JSON để bỏ các entry khó.

Widget vẫn nhận thao tác sau khi cell có dấu ✓. Nếu mất kết nối runtime, dừng thao tác và kết nối
lại rồi chạy từ Cell 1. Chỉ nhãn đã lưu thành công được tính.

## Cell 3 — Kiểm tra tiến độ đã lưu · tùy lúc cần

Chạy để đọc lại file trên Drive và xem số nhãn xác nhận theo từng publication.
Entry đã xem hoặc bỏ qua không được tính. Bảng hiển thị riêng số xác nhận/được chọn/chưa gán.
Nếu chưa mở bảng, cell chỉ nhắc bạn điền biểu mẫu, không báo NameError.

In [ ]:
if 'tool' not in globals():
    print('Chạy Cell 1 và Cell 2 trước, rồi điền biểu mẫu để mở bảng.')
else:
    tool.show_progress()
    display(tool.report_area)

## Cell 4 — Kết thúc phiên và bàn giao

Chạy để đóng bảng, đọc lại tiến độ và hiển thị đường dẫn file cần gửi.
Mỗi lần chọn đã lưu ngay, không cần lưu nhãn thêm hoặc chạy lại auto.

**Bàn giao:** file nhãn Parquet và kế hoạch JSON tương ứng (nếu manual), kèm phiên bản dữ liệu.
Không chỉ gửi ảnh thống kê. Nhãn cũ giữ đúng reviewer/thời gian đã lưu. Các file cấu hình chỉ để
tiện mở lại; không phải ground truth cuối. Không upload dataset/nhãn/cấu hình cá nhân lên GitHub.

Các thành viên chạy độc lập; khi tổng hợp cần đối soát fingerprint, manual không trùng publication
với nhau hoặc với auto, đủ 20 bài và ≥400 nhãn cho nhóm 4 người. Không nối các file khác snapshot
rồi bỏ kiểm tra fingerprint. Chia tập seed 42 và nghiệm thu quota toàn nhóm là bước sau.

In [ ]:
if 'tool' not in globals():
    print('Chưa mở công cụ; không có phiên để kết thúc.')
else:
    tool.finish()
    display(tool.status)
    display(tool.report_area)

## Khi gặp lỗi

| Hiện tượng | Cách xử lý trong luồng thống nhất |
|---|---|
| Source cũ / thiếu module | Chạy Cell 1; mở bản notebook mới từ nhánh M3_E1. Không sửa import từng cell. |
| Không tìm thấy file | Chọn lại bằng Chọn… hoặc Copy path trong biểu mẫu. Kiểm tra đúng tài khoản Drive đã mount. |
| Không biết ROOT ở đâu | Không dùng ROOT để cấu hình. Mọi đường dẫn chọn trong biểu mẫu. |
| Chưa có kế hoạch | Người mới chọn Bắt đầu manual trên dữ liệu riêng. Người có nhãn chọn Tiếp tục manual với kế hoạch gốc. |
| Snapshot khác | Chọn đúng dữ liệu/plan/auto run gốc; không sửa hoặc bỏ fingerprint. |
| Không ghi được Drive | Kiểm tra quyền/dung lượng. Người mới chọn nơi lưu nhãn trên Drive riêng; người có nhãn cần giữ/copy tiến độ có kiểm tra. |
| Lock / entry đổi ở phiên khác | Dừng phiên trùng. Chỉ xóa lock cũ khi chắc chắn writer cũ đã dừng; Nạp lại tiến độ trước khi sửa. |
| Nút không phản hồi / chỉ thấy chữ VBox | Kiểm tra kết nối và chạy lại Cell 1 → Cell 2. GitHub preview không chạy widget; dùng Colab/Jupyter. |
| Git pull lỗi | Giữ bản source sửa để kiểm tra; không reset/force. Có thể bắt đầu runtime Colab mới nếu code chỉ nằm ở runtime và nhãn đã lưu Drive. |
| Thiếu title/authors/DOI | Mở nguồn để xác minh hoặc để pending; không chọn Không có match chỉ vì thiếu trường. |

Không cần chỉnh code theo tên hay số thứ tự người dùng. Mỗi người nhập tên/mã của mình và
chọn dữ liệu/nơi ghi trong cùng biểu mẫu; nhãn đã lưu sẽ được nạp lại khi mở đúng phiên.